<a href="https://colab.research.google.com/github/jetnipitbank-maker/thai-bank-sentiment-analysis/blob/main/04_auto_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL-00] Install all necessary libraries
!pip install -q google-generativeai pydantic tqdm pandas openpyxl matplotlib seaborn pythainlp

In [ ]:
# [CELL-01] อัปเดต: กรองขยะจัดเต็ม + แสดงสถิติเชิงลึก (Print Stats)
import pandas as pd
import json
import glob
import os
import re

def prepare_and_split_dataset(input_folder, seed_output_file, remaining_output_file, sample_size=5000):
    print(f"🔄 เริ่มค้นหาไฟล์ในโฟลเดอร์: '{input_folder}' ...\n")

    # 🔍 1. ค้นหาไฟล์ .jsonl ทั้งหมด
    search_pattern = os.path.join(input_folder, "*.jsonl")
    file_list = glob.glob(search_pattern)

    if not file_list:
        print(f"❌ ไม่พบไฟล์ .jsonl ในโฟลเดอร์ '{input_folder}'")
        return

    print(f"📂 พบไฟล์ทั้งหมด {len(file_list)} ไฟล์")
    all_data = []

    # 📥 2. อ่านไฟล์มารวมกัน
    for file in file_list:
        try:
            with open(file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                for line in lines:
                    all_data.append(json.loads(line.strip()))
        except Exception as e:
            print(f"❌ Error อ่านไฟล์ {file}: {e}")

    if not all_data:
        print("ไม่มีข้อมูลให้ดำเนินการต่อ")
        return

    df = pd.DataFrame(all_data)
    print(f"\n📊 รวมข้อมูลดิบทั้งหมดได้: {len(df):,} ข่าว")

    # 🧹 3. ลบข้อมูลซ้ำ
    df_cleaned = df.drop_duplicates(subset=['url', 'related_banks']).copy()
    duplicates_removed = len(df) - len(df_cleaned)
    print(f"🧹 ลบข่าวที่ซ้ำกันออกไป (Duplicate URL/Bank): {duplicates_removed:,} ข่าว")

    # 🛑 4. กรองข่าวที่ไม่เกี่ยวข้อง (Blacklist Filter) จัดกลุ่มเพื่อเก็บสถิติ
    blacklist_groups = {
        "ไม่ใช่ธนาคาร (ชื่อคล้าย)": [
            "สายการบินกรุงเทพ", "บางกอกแอร์เวย์ส", "มหาวิทยาลัยกรุงเทพ", "กรุงเทพมหานคร", "กทม.",
            "โรงพยาบาลกรุงเทพ", "กรุงเทพประกัน", "BDMS", "BKI", "BLA", "ทางด่วนกรุงเทพ", "รถไฟฟ้ากรุงเทพ",
            r"(?<!ธนาคาร)กรุงเทพฯ",
            r"(?<!ธนาคาร)กรุงเทพ"
        ],
        "ศูนย์วิจัย/เศรษฐกิจมหภาค": [
            "ศูนย์วิจัย", "วิจัยกสิกรไทย", "วิจัยกรุงศรี", "ttb analytics", "EIC", "อีไอซี", "วิจัยเศรษฐกิจ", "สภาพัฒน์", "ค่าเงินบาท"
        ],
        "สรุปภาวะตลาด/สถิติรายวัน": [
            "สรุปหุ้น", "ขายชอร์ต", "ชอร์ตเซล", "หุ้นไทยแนวโน้ม", "หุ้นไทยปิด", "หุ้นไทยลุ้น", "ภาวะตลาดหุ้น", "สรุปการซื้อขาย", "NVDR", "บิ๊กล็อต", "Big Lot"
        ],
        "บริษัทในเครือ (บล./บลจ.)": [
            r"บล\.", "บริษัทหลักทรัพย์", r"บลจ\.", "กองทุนรวม"
        ]
    }

    print("\n🗑️ เริ่มสแกนและเตะข่าวขยะออก (แยกตามหมวดหมู่)...")
    total_irrelevant = 0

    for group_name, keywords in blacklist_groups.items():
        pattern = '|'.join(keywords)

        # ตรวจสอบ Title หรือ Content
        mask_title = df_cleaned['title'].fillna('').str.contains(pattern, case=False, regex=True)
        mask_content = df_cleaned['content'].fillna('').str.contains(pattern, case=False, regex=True)
        mask_combined = mask_title | mask_content

        drop_count = mask_combined.sum()
        total_irrelevant += drop_count

        print(f"   - เตะหมวด '{group_name}': {drop_count:,} ข่าว")

        # อัปเดต df ให้เหลือเฉพาะตัวที่รอด
        df_cleaned = df_cleaned[~mask_combined].copy()

    print(f"🔴 รวมข่าวขยะที่ถูกเตะออกทั้งหมด: {total_irrelevant:,} ข่าว")
    print(f"✨ ข่าวที่บริสุทธิ์และใช้งานได้จริง: {len(df_cleaned):,} ข่าว")

    # 📈 5. แสดงสถิติข่าวรายธนาคาร (หลัง Clean)
    print("\n🏦 สถิติข่าวรายธนาคาร (พร้อมใช้งาน):")
    bank_counts = df_cleaned['related_banks'].value_counts()
    for bank, count in bank_counts.items():
        print(f"   - {bank}: {count:,} ข่าว")

    # 📋 6. จัดเรียงลำดับคอลัมน์ใหม่
    desired_cols = ['related_banks', 'title', 'content', 'date', 'url']
    existing_cols = [col for col in desired_cols if col in df_cleaned.columns]
    df_cleaned = df_cleaned[existing_cols]

    # 🎲 7. แบ่งข้อมูล (Split Dataset)
    actual_sample_size = min(sample_size, len(df_cleaned))

    # ส่วนที่ 1: Seed Dataset (สุ่มออกมา)
    df_seed = df_cleaned.sample(n=actual_sample_size, random_state=99)

    # ส่วนที่ 2: Remaining Clean Dataset (เอาข้อมูลทั้งหมด มาหักลบ Seed ออกไป)
    df_remaining = df_cleaned.drop(df_seed.index)

    # 💾 8. บันทึกไฟล์แยก 2 ส่วน
    df_seed.to_json(seed_output_file, orient='records', lines=True, force_ascii=False)
    df_remaining.to_json(remaining_output_file, orient='records', lines=True, force_ascii=False)

    print("\n" + "="*50)
    print("🎯 สรุปผลการแบ่งไฟล์ (Split Summary):")
    print("="*50)
    print(f"1️⃣ [ส่วนทำ Seed] ข้อมูลสำหรับส่งให้ AI ช่วย Label")
    print(f"   👉 จำนวน: {len(df_seed):,} ข่าว")
    print(f"   📁 บันทึกที่: {seed_output_file}")

    print(f"\n2️⃣ [ส่วน Clean] ข้อมูลบริสุทธิ์ที่รอโมเดลมาทำนาย (Inference)")
    print(f"   👉 จำนวน: {len(df_remaining):,} ข่าว")
    print(f"   📁 บันทึกที่: {remaining_output_file}")
    print("="*50)

if __name__ == "__main__":
    INPUT_FOLDER = r"C:\Users\USER\Desktop\BANKS_News\Final\ForFinetuning"
    SEED_FILE = "01_seed_dataset_for_labeling.jsonl"
    REMAINING_FILE = "02_remaining_clean_dataset.jsonl"

    prepare_and_split_dataset(INPUT_FOLDER, SEED_FILE, REMAINING_FILE, sample_size=5000)

In [ ]:
# [CELL-05] ฉบับเริ่มไฟล์ใหม่ (v2) + ปรับลำดับคอลัมน์ + [-99]
import json
import time
import os
import google.generativeai as genai
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
from google.generativeai.types import HarmCategory, HarmBlockThreshold

# ==========================================
# 1. ตั้งค่า API และชื่อไฟล์ใหม่
# ==========================================
API_KEY = "api_key"
INPUT_FILE = "01_seed_dataset_for_labeling.jsonl"
# เปลี่ยนชื่อไฟล์ที่นี่เพื่อให้เริ่มเก็บข้อมูลใหม่ ไม่ปนกับของเดิม
OUTPUT_FILE = "seed_dataset_labeled_v3.jsonl"
BATCH_SIZE = 15

genai.configure(api_key=API_KEY)

# ==========================================
# 2. โครงสร้าง JSON
# ==========================================
class SingleSentiment(BaseModel):
    id: str = Field(description="รหัสอ้างอิงของข่าว")
    target_bank: str = Field(description="ชื่อย่อธนาคารเป้าหมาย")
    reasoning: str = Field(description="เหตุผล 1-2 ประโยคอ้างอิงหลักการเงิน")
    sentiment: int = Field(description="คะแนน 1 (Pos), 0 (Neu), -1 (Neg), หรือ -99 (Irrelevant)")
    confidence_score: float = Field(description="ความมั่นใจ 0.00 ถึง 1.00")

class BatchSentimentResult(BaseModel):
    results: list[SingleSentiment] = Field(description="รายการผลลัพธ์การวิเคราะห์")

# ==========================================
# 3. System Prompt (รองรับ -99 สำหรับข่าวไม่เกี่ยวข้อง)
# ==========================================
SYSTEM_PROMPT = """
Role: คุณคือนักวิเคราะห์เชิงปริมาณ (Quantitative Analyst) และผู้เชี่ยวชาญด้านปัจจัยพื้นฐานของหุ้นกลุ่มธนาคารพาณิชย์ขนาดใหญ่ (D-SIBs) ในไทย

Rules for Sentiment Scoring:
[ 1 ] Positive: ข่าวที่มีผลกระทบเชิงบวกชัดเจน เช่น กำไรโต, NPL ลด, ปันผลสูง
[ 0 ] Neutral: ข่าวภาพรวมตลาด, ข่าวที่ไม่มีผลกระทบงบดุลชัดเจน, หรือข่าวดำเนินงานทั่วไป
[-1 ] Negative: ข่าวที่มีผลกระทบเชิงลบชัดเจน เช่น กำไรลด, ตั้งสำรองเพิ่ม, NIM หด
[-99] Irrelevant: ข่าวที่ไม่เกี่ยวข้องกับธนาคารเป้าหมายเลย (เช่น เป็นชื่อศูนย์วิจัย, ชื่อสายการบิน หรือบริษัทอื่นที่ชื่อคล้ายกัน)

Instructions:
วิเคราะห์แต่ละข่าวแยกกันตาม ID และส่งคืนผลลัพธ์เป็น Array ภายใต้ Key 'results' โดยต้องตอบผลลัพธ์ให้ครบทุก ID
"""

model = genai.GenerativeModel(
    model_name="gemini-3.1-flash-lite",
    system_instruction=SYSTEM_PROMPT
)

safety_settings = {
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
}

# ==========================================
# 4. ฟังก์ชันวิเคราะห์ Batch
# ==========================================
def analyze_news_batch(news_batch, retries=3, batch_num=1):
    prompt_lines = ["โปรดวิเคราะห์ข่าวต่อไปนี้:\n"]
    for item in news_batch:
        prompt_lines.append(f"--- ID: {item['id']} ---")
        prompt_lines.append(f"Target_Bank: {item['bank']}")
        prompt_lines.append(f"Title: {item['title']}")
        prompt_lines.append(f"Content: {item['content']}\n")

    prompt = "\n".join(prompt_lines)

    for attempt in range(retries):
        try:
            response = model.generate_content(
                prompt,
                safety_settings=safety_settings,
                generation_config=genai.GenerationConfig(
                    response_mime_type="application/json",
                    response_schema=BatchSentimentResult,
                    temperature=0.1
                )
            )
            return json.loads(response.text).get("results", [])
        except Exception as e:
            print(f"\n   ⚠️ [Batch {batch_num}] Error: {e}")
            if attempt < retries - 1:
                time.sleep(5)
            else:
                return None

# ==========================================
# 5. Main Loop (Resume ภายใน v2 เท่านั้น)
# ==========================================
def create_seed_dataset():
    processed_keys = set()
    if os.path.exists(OUTPUT_FILE):
        print(f"🔍 ตรวจสอบความคืบหน้าในไฟล์ {OUTPUT_FILE}...")
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line.strip())
                    key = f"{data.get('url')}_{data.get('related_banks')}"
                    processed_keys.add(key)
                except: continue
        print(f"✅ ข้ามข่าวที่ทำไปแล้วในไฟล์ใหม่นี้ {len(processed_keys)} ข่าว")
    else:
        print(f"📁 ไม่พบไฟล์ {OUTPUT_FILE} เดิม จะเริ่มสร้างไฟล์ใหม่ตั้งแต่ข่าวแรก...")

    raw_data = []
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            data = json.loads(line.strip())
            key = f"{data.get('url')}_{data.get('related_banks')}"
            # เช็คเฉพาะในไฟล์ v2
            if key not in processed_keys:
                raw_data.append({
                    "id": str(i),
                    "date": data.get("date"),
                    "url": data.get("url"),
                    "bank": data.get("related_banks"),
                    "title": data.get("title"),
                    "content": data.get("content")
                })

    if not raw_data:
        print("🎉 ทุกข่าวในไฟล์นี้ถูก Label ครบถ้วนแล้ว!")
        return

    total_batches = (len(raw_data) // BATCH_SIZE) + (1 if len(raw_data) % BATCH_SIZE != 0 else 0)
    print(f"📊 เริ่มทำงานต่ออีก {len(raw_data)} ข่าว ({total_batches} Batches)")

    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
        for i in range(0, len(raw_data), BATCH_SIZE):
            batch = raw_data[i : i + BATCH_SIZE]
            batch_num = (i // BATCH_SIZE) + 1
            print(f"⏳ Batch {batch_num}/{total_batches}...", end=" ")

            ai_results = analyze_news_batch(batch, batch_num=batch_num)

            if ai_results:
                result_map = {str(res['id']): res for res in ai_results}
                for item in batch:
                    res = result_map.get(str(item['id']))
                    if res:
                        # ✨ จัดลำดับผลลัพธ์ไว้หน้าสุด
                        final_record = {
                            "sentiment": res.get("sentiment"),
                            "reasoning": res.get("reasoning"),
                            "confidence_score": res.get("confidence_score"),
                            "related_banks": item["bank"],
                            "title": item["title"],
                            "content": item["content"],
                            "date": item["date"],
                            "url": item["url"]
                        }
                        f_out.write(json.dumps(final_record, ensure_ascii=False) + "\n")
                print("✅")
            else:
                print("❌")
            time.sleep(5)

    print("\n" + "="*50)
    print(f"🎉 สร้าง Seed Dataset (v2) สำเร็จ!")
    print(f"📁 บันทึกผลลัพธ์ไว้ที่: {OUTPUT_FILE}")
    print("="*50)

if __name__ == "__main__":
    create_seed_dataset()

In [ ]:
# [CELL-06]
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

def summarize_labeled_data(file_path):
    data = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line.strip()))
    except FileNotFoundError:
        print(f"❌ ไม่พบไฟล์: {file_path}")
        return

    df = pd.DataFrame(data)

    if df.empty:
        print("⚠️ ไฟล์ว่างเปล่า ไม่มีข้อมูลให้สรุป")
        return

    # 1. สรุปจำนวน Label
    sentiment_map = {1: 'Positive', 0: 'Neutral', -1: 'Negative'}
    df['sentiment_label'] = df['sentiment'].map(sentiment_map)

    counts = df['sentiment_label'].value_counts().reindex(['Positive', 'Neutral', 'Negative']).fillna(0)
    pcts = df['sentiment_label'].value_counts(normalize=True).reindex(['Positive', 'Neutral', 'Negative']).fillna(0) * 100

    # 2. สรุปค่า Confidence Score
    conf_stats = df.groupby('sentiment_label')['confidence_score'].agg(['mean', 'std', 'min', 'max'])

    print("="*50)
    print("📊 สรุปผลการ Label ข่าว (Sentiment Statistics)")
    print("="*50)
    print(f"จำนวนข่าวทั้งหมด: {len(df):,} ข่าว\n")

    summary_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.map('{:.2f}%'.format)})
    display(summary_df)

    print("\n💡 ค่าความเชื่อมั่น (Confidence Score Analysis):")
    display(conf_stats.round(4))

    # 3. Visualization
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))

    # Bar Plot
    sns.barplot(x=counts.index, y=counts.values, palette=['#2ecc71', '#bdc3c7', '#e74c3c'], ax=ax[0])
    ax[0].set_title('Distribution of News Sentiment')
    ax[0].set_ylabel('Number of News')

    # Box Plot for Confidence
    sns.boxplot(x='sentiment_label', y='confidence_score', data=df,
                palette=['#2ecc71', '#bdc3c7', '#e74c3c'], ax=ax[1])
    ax[1].set_title('Confidence Score Distribution by Label')
    ax[1].set_ylim(0, 1.1)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    LABELED_FILE = "seed_dataset_labeled_v3.jsonl"
    summarize_labeled_data(LABELED_FILE)

In [ ]:
# [CELL-07]
import pandas as pd
import json

def inspect_sentiment_samples(file_path, n_samples=5):
    data = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line.strip()))
    except FileNotFoundError:
        print(f"❌ ไม่พบไฟล์: {file_path}")
        return

    df = pd.DataFrame(data)

    # ตัดกลุ่ม -99 ออกก่อนเพื่อดูเฉพาะกลุ่มที่จะใช้ Train
    df_train = df[df['sentiment'] != -99].copy()

    sentiment_map = {1: 'Positive', 0: 'Neutral', -1: 'Negative'}

    print(f"🔍 ตรวจสอบตัวอย่างข่าวที่มี Confidence ต่ำสุด แยกตามกลุ่ม (จากไฟล์ v3)")
    print("="*80)

    for sent_val, sent_name in sentiment_map.items():
        subset = df_train[df_train['sentiment'] == sent_val].sort_values(by='confidence_score', ascending=True).head(n_samples)

        print(f"\n>>> กลุ่ม: {sent_name} (แสดง {len(subset)} ตัวอย่างที่ก้ำกึ่งที่สุด) <<<")
        print("-"*80)

        for i, row in subset.iterrows():
            print(f"📌 ID: {i} | ⭐ Confidence: {row.get('confidence_score')}")
            print(f"🏦 Bank: {row.get('related_banks')} | 💡 Reasoning: {row.get('reasoning')}")
            print(f"📰 Title: {row.get('title')}")
            print(f"📄 Content: {row.get('content')[:200]}...")
            print("."*40)

if __name__ == '__main__':
    # เรียกใช้ฟังก์ชันตรวจสอบข้อมูลชุดล่าสุด
    inspect_sentiment_samples('seed_dataset_labeled_v3.jsonl')

In [ ]:
# [CELL-08]
import pandas as pd

# 1. โหลดข้อมูลจากไฟล์ v3
file_path = "seed_dataset_labeled_v3.jsonl"
df = pd.read_json(file_path, lines=True)

# 2. กรองเฉพาะข่าวที่มี Sentiment เป็น 1, 0, -1 (ตัด -99 ออก)
# เราจะ overwrite ตัวแปร df เพื่อให้ state ล่าสุดสะอาด 100%
initial_count = len(df)
df = df[df['sentiment'].isin([1, 0, -1])].copy()
removed_count = initial_count - len(df)

# 3. บันทึกไฟล์ใหม่สำหรับใช้ Fine-tuning โดยเฉพาะ
output_clean_file = "seed_dataset_final_train.jsonl"
df.to_json(output_clean_file, orient='records', lines=True, force_ascii=False)

print(f"✅ ดำเนินการกรองข้อมูลเสร็จสิ้น")
print(f"🗑️ จำนวนข่าวขยะ (-99) ที่ถูกเตะออก: {removed_count:,} ข่าว")
print(f"✨ จำนวนข้อมูลที่สะอาดพร้อม Train (df): {len(df):,} ข่าว")
print(f"📁 บันทึกไฟล์พร้อมใช้ที่: {output_clean_file}")

# ตรวจสอบเพื่อความมั่นใจ
print("\n📊 เช็คยอด Sentiment อีกครั้ง:")
print(df['sentiment'].value_counts())